In [1]:
import pandas as pd
import re

def parse_can_log(file_path):
    """Parse CAN log file and return a pandas DataFrame"""
    data = []
    
    with open(file_path, 'r') as f:
        for line in f:
            # Extract fields using regex
            match = re.search(r'Timestamp:\s([\d.]+)\s+ID:\s(\w+)\s+(\w+)\s+DLC:\s(\d+)\s+(.*)', line)
            if match:
                timestamp, can_id, flags, dlc, hex_data = match.groups()
                data.append({
                    'Timestamp': float(timestamp),
                    'ID': can_id,
                    'Flags': flags,
                    'DLC': int(dlc),
                    'Data': hex_data.strip()
                })
    
    df = pd.DataFrame(data)
    return df

# Usage
df = parse_can_log(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\normal_run_data\normal_run_data.txt')
print(df)

           Timestamp    ID Flags  DLC                     Data
0       1.479121e+09  0350   000    8  05 28 84 66 6d 00 00 a2
1       1.479121e+09  02c0   000    8  14 00 00 00 00 00 00 00
2       1.479121e+09  0430   000    8  00 00 00 00 00 00 00 00
3       1.479121e+09  04b1   000    8  00 00 00 00 00 00 00 00
4       1.479121e+09  01f1   000    8  00 00 00 00 00 00 00 00
...              ...   ...   ...  ...                      ...
988866  1.479122e+09  02b0   000    5           ac 05 0c 07 7f
988867  1.479122e+09  0316   000    8  05 38 10 0c 38 28 01 7a
988868  1.479122e+09  018f   000    8  fe 31 00 00 00 4b 00 00
988869  1.479122e+09  0260   000    8  32 38 39 30 ff 93 59 1c
988870  1.479122e+09  02a0   000    8  20 00 75 1d 01 04 dd 00

[988871 rows x 5 columns]


In [2]:
def parse_can_csv(file_path):
    # try reading with a header first
    df_DoS = pd.read_csv(file_path, skipinitialspace=True)
    
    if 'ID dlc' in df_DoS.columns:
        df_DoS[['ID', 'DLC']] = df_DoS['ID dlc'].astype(str).str.split(r'\s+', n=1, expand=True)
        df_DoS = df_DoS.drop(columns=['ID dlc'])
    elif set(['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','attack/nonattack']).issubset(df_DoS.columns):
        df_DoS = df_DoS.rename(columns={'attack/nonattack': 'Attack'})
    else:
        names = ['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','Attack']
        df_DoS = pd.read_csv(file_path, header=None, names=names, skipinitialspace=True)

    data_cols = [c for c in df_DoS.columns if c.lower().startswith('data')]
    df_DoS['Data'] = df_DoS[data_cols].astype(str).apply(
        lambda row: ' '.join(x for x in row if x not in ['', 'nan', 'NaN']),
        axis=1
    ).str.strip()

    df_DoS['Timestamp'] = pd.to_numeric(df_DoS['Timestamp'], errors='coerce')
    df_DoS['DLC'] = pd.to_numeric(df_DoS['DLC'], errors='coerce').astype('Int64')

    return df_DoS[['Timestamp','ID','DLC','Data','Attack']]




In [3]:
def parse_can_csv(file_path):
    df_can = pd.read_csv(file_path, skipinitialspace=True)

    if 'ID dlc' in df_can.columns:
        df_can[['ID', 'DLC']] = df_can['ID dlc'].astype(str).str.split(r'\s+', n=1, expand=True)
        df_can = df_can.drop(columns=['ID dlc'])
    elif set(['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','attack/nonattack']).issubset(df_can.columns):
        df_can = df_can.rename(columns={'attack/nonattack': 'Attack'})
    else:
        names = ['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','Attack']
        df_can = pd.read_csv(file_path, header=None, names=names, skipinitialspace=True)

    unnamed = [c for c in df_can.columns if isinstance(c, str) and c.startswith('Unnamed')]
    if 'Attack' not in df_can.columns and unnamed:
        df_can = df_can.rename(columns={unnamed[-1]: 'Attack'})

    data_cols = sorted(
        [c for c in df_can.columns if isinstance(c, str) and c.lower().startswith('data')],
        key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 0
    )

    def build_data(row):
        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        values = []
        for col in data_cols[:min(dlc, len(data_cols))]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            values.append(s)

        return ' '.join(values).strip()

    def infer_attack(row):
        attack = row.get('Attack')
        if isinstance(attack, str) and attack.strip():
            return attack

        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        for col in data_cols[min(dlc, len(data_cols)):]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            if not re.fullmatch(r'[0-9A-Fa-f]{2}', s):
                return s

        return pd.NA

    if data_cols:
        df_can['Data'] = df_can.apply(build_data, axis=1)
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA
        df_can['Attack'] = df_can.apply(infer_attack, axis=1).combine_first(df_can['Attack'])
    else:
        df_can['Data'] = df_can.get('Data', '').astype(str).str.strip()
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA

    df_can['Timestamp'] = pd.to_numeric(df_can['Timestamp'], errors='coerce')
    df_can['DLC'] = pd.to_numeric(df_can['DLC'], errors='coerce').astype('Int64')

    return df_can[['Timestamp','ID','DLC','Data','Attack']]

In [4]:
df_DoS = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\DoS_dataset.csv')
print(df_DoS.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478198e+09  0316    8  05 21 68 09 21 21 00 6f      R
1  1.478198e+09  018f    8  fe 5b 00 00 00 3c 00 00      R
2  1.478198e+09  0260    8  19 21 22 30 08 8e 6d 3a      R
3  1.478198e+09  02a0    8  64 00 9a 1d 97 02 bd 00      R
4  1.478198e+09  0329    8  40 bb 7f 14 11 20 00 14      R


In [5]:
df_Fuzzy = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\Fuzzy_dataset.csv')
print(df_Fuzzy.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478196e+09  0545    8  d8 00 00 8a 00 00 00 00      R
1  1.478196e+09  02b0    5           ff 7f 00 05 49      R
2  1.478196e+09  0002    8  00 00 00 00 00 01 07 15      R
3  1.478196e+09  0153    8  00 21 10 ff 00 ff 00 00      R
4  1.478196e+09  0130    8  19 80 00 ff fe 7f 07 60      R


In [6]:
df_gear = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\gear_dataset.csv')
print(df_gear.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478193e+09  0140    8  00 00 00 00 10 29 2a 24      R
1  1.478193e+09  02c0    8  15 00 00 00 00 00 00 00      R
2  1.478193e+09  0350    8  05 20 44 68 77 00 00 7e      R
3  1.478193e+09  0370    8  00 20 00 00 00 00 00 00      R
4  1.478193e+09  043f    8  10 40 60 ff 78 c4 08 00      R


In [7]:
df_RPM = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\RPM_dataset.csv')
print(df_RPM.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478191e+09  0316    8  05 22 68 09 22 20 00 75      R
1  1.478191e+09  018f    8  fe 3b 00 00 00 3c 00 00      R
2  1.478191e+09  0260    8  19 22 22 30 ff 8f 6e 3f      R
3  1.478191e+09  02a0    8  60 00 83 1d 96 02 bd 00      R
4  1.478191e+09  0329    8  dc b8 7e 14 11 20 00 14      R


In [8]:

# Normalize the normal dataset and add an Attack label of 'R'
df_normal = df[['Timestamp', 'ID', 'DLC', 'Data']].copy()
df_normal['Attack'] = 'R'

# Combine all datasets into one dataframe
df_combined = pd.concat(
    [
        df_normal,
        df_DoS[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_Fuzzy[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_gear[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_RPM[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
    ],
    ignore_index=True
)

# Optional: keep columns in a consistent order
df_combined = df_combined[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']]

print(df_combined.head())
print(df_combined['Attack'].value_counts())

      Timestamp    ID  DLC                     Data Attack
0  1.479121e+09  0350    8  05 28 84 66 6d 00 00 a2      R
1  1.479121e+09  02c0    8  14 00 00 00 00 00 00 00      R
2  1.479121e+09  0430    8  00 00 00 00 00 00 00 00      R
3  1.479121e+09  04b1    8  00 00 00 00 00 00 00 00      R
4  1.479121e+09  01f1    8  00 00 00 00 00 00 00 00      R
Attack
R    15226829
T     2331517
Name: count, dtype: int64


In [9]:
del df, df_normal, df_DoS, df_Fuzzy, df_gear, df_RPM



In [10]:
import pandas as pd

# Only keep the columns you need
X = df_combined.loc[:8779174, ["ID", "DLC", "Data"]].copy()

# Convert hexadecimal CAN IDs directly to uint32
X["ID"] = X["ID"].map(lambda x: int(x, 16)).astype("uint32")

# DLC only needs values 0-8
X["DLC"] = X["DLC"].astype("uint8")

# Convert payload into a 64-bit integer
X["Data"] = (
    X["Data"]
    .str.replace(" ", "", regex=False)
    .map(lambda x: int(x, 16))
    .astype("uint64")
)

# Labels
y = df_combined["Attack"][:8779175].astype("category")

print(X.head())

     ID  DLC                 Data
0   848    8   371692544708313250
1   704    8  1441151880758558720
2  1072    8                    0
3  1201    8                    0
4   497    8                    0


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict
predictions = rf.predict(X_test)

In [12]:
predictions
y_test

7679477    R
1040929    R
518981     R
7244829    R
1290334    R
          ..
557338     R
3190796    R
5260355    R
650488     R
1043203    R
Name: Attack, Length: 1755835, dtype: category
Categories (2, str): ['R', 'T']

In [13]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9982


In [14]:
from sklearn.tree import export_text

print("Random Forest summary:")
print("n_estimators:", rf.n_estimators)
print("max_depth:", rf.max_depth)
print("min_samples_leaf:", rf.min_samples_leaf)
print("feature_importances:", dict(zip(["ID", "DLC", "Data"], rf.feature_importances_)))
print()

print("First tree structure (truncated):")
print(export_text(rf.estimators_[49], feature_names=["ID", "DLC", "Data"], max_depth=10))

Random Forest summary:
n_estimators: 10
max_depth: None
min_samples_leaf: 1
feature_importances: {'ID': np.float64(0.597320422130698), 'DLC': np.float64(0.006291551565413817), 'Data': np.float64(0.39638802630388825)}

First tree structure (truncated):


IndexError: list index out of range

In [ ]:
import joblib

joblib.dump(rf, "saved_models/random_forest_original.joblib")

['saved_models/random_forest_original.joblib']

In [ ]:
rf_loaded = joblib.load("saved_models/random_forest_original.joblib")
print(rf_loaded)

RandomForestClassifier(n_estimators=10, random_state=42)


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average="weighted", zero_division=0)
recall = recall_score(y_test, predictions, average="weighted", zero_division=0)
f1 = f1_score(y_test, predictions, average="weighted", zero_division=0)

labels = list(rf.classes_)
cm = confusion_matrix(y_test, predictions, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

fpr = {}
fnr = {}
for i, label in enumerate(labels):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    fpr[label] = fp / (fp + tn) if (fp + tn) != 0 else 0.0
    fnr[label] = fn / (fn + tp) if (fn + tp) != 0 else 0.0

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted precision: {precision:.4f}")
print(f"Weighted recall: {recall:.4f}")
print(f"Weighted F1 score: {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, labels=labels, zero_division=0))
print("Confusion matrix:")
print(cm_df)
print("\nFalse positive rate per class:")
for label, rate in fpr.items():
    print(f"  {label}: {rate:.4f}")
print("\nFalse negative rate per class:")
for label, rate in fnr.items():
    print(f"  {label}: {rate:.4f}")

Accuracy: 0.9982
Weighted precision: 0.9982
Weighted recall: 0.9982
Weighted F1 score: 0.9982

Classification report:
              precision    recall  f1-score   support

           R       1.00      1.00      1.00   1529629
           T       0.99      1.00      0.99    226206

    accuracy                           1.00   1755835
   macro avg       0.99      1.00      1.00   1755835
weighted avg       1.00      1.00      1.00   1755835

Confusion matrix:
         R       T
R  1526657    2972
T      193  226013

False positive rate per class:
  R: 0.0009
  T: 0.0019

False negative rate per class:
  R: 0.0019
  T: 0.0009
